# Text + image fusion (PhoBERT + CLIP ViT-B/16) - 5-fold cross-validation (Kaggle)

**Before running**: Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).

One model reads the title + lead with PhoBERT and the thumbnail with CLIP, concatenates the two features and classifies. Same 5 folds as every earlier run, so the out-of-fold scores are directly comparable:

| reference | out-of-fold F1 |
|---|---|
| text only (PhoBERT) | 0.771 |
| image only (CLIP, top 4 layers tuned) | 0.571 |

| run | what differs |
|---|---|
| `fusion` | CLIP frozen, auxiliary text-only and image-only heads (weight 0.3 each) |
| `fusion_no_aux` | the same without the auxiliary heads |

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_fusion_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_fusion_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/fusion_cv")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "fusion_cv"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train

Roughly 12-15 minutes per run on a T4. Remove an entry from `RUNS` to skip it; uncomment the last one to also fine-tune CLIP's top layers (slower).

In [ ]:
from trainer.train_fusion_cv import FusionConfig, run

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    max_length=128,
    use_lead=True,
    image_size=224,
    augment=True,
    batch_size=32,
    lr=2e-5,              # PhoBERT
    layer_decay=0.9,
    head_lr=1e-3,         # fusion head and auxiliary heads
    epochs=5,
    weighted_loss=True,
    patience=3,
    monitor="loss",
    seed=42,
    num_workers=4 if ON_KAGGLE else 0,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
RUNS = {
    "fusion": dict(text_aux_weight=0.3, image_aux_weight=0.3),
    "fusion_no_aux": dict(text_aux_weight=0.0, image_aux_weight=0.0),
    # "fusion_tune_clip": dict(text_aux_weight=0.3, image_aux_weight=0.3, unfreeze_image_layers=4, image_lr=1e-5),
}

results = {}
for name, overrides in RUNS.items():
    print(f"################ {name} ################")
    results[name] = run(FusionConfig(**BASE, **overrides, out_dir=str(OUT_DIR / name)))

## Compare with text only and image only

Judge by `oof_f1` (3,067 articles). Rows ending in `head` are the auxiliary heads inside the fusion model: what each branch can do alone after joint training.

In [ ]:
import json
import numpy as np
import pandas as pd
from trainer.trainer import classification_metrics

TEXT_DIR = CODE_DIR / "outputs" / "text_baseline"      # PhoBERT, same folds
text = None
if (TEXT_DIR / "oof_predictions.csv").exists():
    text = {"oof": pd.read_csv(TEXT_DIR / "oof_predictions.csv"), "test": pd.read_csv(TEXT_DIR / "test_predictions.csv"),
            "summary": json.loads((TEXT_DIR / "summary.json").read_text())}

def row(s):
    return {"oof_f1": s["oof"]["f1"], "oof_precision": s["oof"]["precision"], "oof_recall": s["oof"]["recall"],
            "single_test_f1": s["single_model_test_mean"]["f1"], "ensemble_test_f1": s["ensemble_test"]["f1"]}

rows = {}
if text:
    rows["text only (PhoBERT)"] = row(text["summary"])
rows["image only (CLIP), earlier run"] = {"oof_f1": 0.5712, "oof_precision": 0.4854, "oof_recall": 0.6938,
                                          "single_test_f1": 0.5974, "ensemble_test_f1": 0.6047}
for name, r in results.items():
    rows[name] = row(r["summary"])
    for branch in ("text", "image"):
        m = r["summary"].get(f"oof_{branch}_head")
        if m:
            rows[f"  {name}: {branch} head"] = {"oof_f1": m["f1"], "oof_precision": m["precision"], "oof_recall": m["recall"]}
comparison = pd.DataFrame(rows).T
display(comparison.round(4))

SHOW = max(results, key=lambda n: results[n]["summary"]["oof"]["f1"])   # run detailed below; set a name to override
result = results[SHOW]
print("showing details for:", SHOW)

## Is fusion really better than text?

Both models predicted the same 3,067 articles out-of-fold, so the difference can be tested directly. If the 95% interval of the F1 difference includes 0, the fusion model is not measurably better than the text model.

In [ ]:
def f1(y, pred):
    tp = (y & pred).sum()
    return 2 * tp / max(2 * tp + (~y & pred).sum() + (y & ~pred).sum(), 1)

if text is None:
    print("outputs/text_baseline not found in the repo - skipping")
else:
    rng = np.random.default_rng(0)
    for name, r in results.items():
        pair = text["oof"][["id", "label", "prob_clickbait"]].merge(
            r["oof_predictions"][["id", "prob_clickbait"]], on="id", suffixes=("_text", "_fusion"), validate="one_to_one")
        y = (pair["label"] == "clickbait").to_numpy()
        t, f = (pair["prob_clickbait_text"] >= 0.5).to_numpy(), (pair["prob_clickbait_fusion"] >= 0.5).to_numpy()
        idx = rng.integers(0, len(y), (2000, len(y)))
        diffs = np.array([f1(y[i], f[i]) - f1(y[i], t[i]) for i in idx])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        print(f"{name}: F1 {f1(y, f):.4f} vs text {f1(y, t):.4f} | difference {f1(y, f) - f1(y, t):+.4f} "
              f"(95% interval {lo:+.4f} to {hi:+.4f})")
        print(f"   articles text got wrong that fusion fixes: {((t != y) & (f == y)).sum()} | "
              f"articles text got right that fusion breaks: {((t == y) & (f != y)).sum()} | "
              f"same decision on {(t == f).mean():.1%}")

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, `tokenizer/`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/fusion_cv/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")